In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Configuración de la simulación
semanas = 25
dias_simulacion = semanas * 7
# 31/12/2023 es Domingo. La semana 1 arranca con recepción.
fecha_inicio = datetime(2023, 12, 31) 
densidad_std = 1.030
nombre_fichero = 'queseria_final_tidy.csv'

# Traducción de días
dias_es = {
    'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles',
    'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}

# Parámetros de variabilidad
media_perdida = 0.04      
sigma_perdida = 0.005
media_agua = 0.005        
sigma_agua = 0.001        

# Objetivo de fabricación
objetivo_fabricacion_base = 5000 

data = []
stock_leche_litros = 0.0
stock_grasa_kg_total = 0.0
stock_proteina_kg_total = 0.0

for i in range(dias_simulacion):
    fecha = fecha_inicio + timedelta(days=i)
    dia_semana_eng = fecha.strftime('%A')
    dia_semana_es = dias_es[dia_semana_eng]
    num_dia = fecha.weekday() # 0=Lunes, 4=Viernes, 6=Domingo
    
    # --- 1. ENTRADAS (Domingo a Jueves) ---
    entrada_kg = 0
    entrada_litros = 0
    grasa_g_l = 0.0
    prot_g_l = 0.0
    
    if num_dia in [6, 0, 1, 2, 3]: # Domingo a Jueves
        entrada_kg = int(round(np.random.normal(5150, 300))) 
        entrada_litros = int(round(entrada_kg / densidad_std))
        grasa_g_l = np.random.normal(37, 1.5)
        prot_g_l = np.random.normal(32, 1.0)
        
        stock_leche_litros += entrada_litros
        stock_grasa_kg_total += (entrada_litros * grasa_g_l) / 1000
        stock_proteina_kg_total += (entrada_litros * prot_g_l) / 1000

    # --- 2. FABRICACIÓN (Lunes a Viernes) ---
    envio_litros_final = 0
    envio_grasa_g_l = 0.0
    envio_prot_g_l = 0.0
    perdida_mg_dia = 0.0
    perdida_mp_dia = 0.0
    inc_agua_dia = 0.0
    
    if num_dia in [0, 1, 2, 3, 4]: # Lunes a Viernes
        if num_dia == 4: # VIERNES: Vaciado total
            envio_litros_teoricos = stock_leche_litros
        else:
            variacion = np.random.uniform(-500, 500)
            envio_litros_teoricos = round(objetivo_fabricacion_base + variacion)
            envio_litros_teoricos = min(envio_litros_teoricos, stock_leche_litros)

        if stock_leche_litros > 0 and envio_litros_teoricos > 0:
            factor_envio = (envio_litros_teoricos / stock_leche_litros)
            kg_grasa_teoricos = stock_grasa_kg_total * factor_envio
            kg_prot_teoricos = stock_proteina_kg_total * factor_envio
        else:
            kg_grasa_teoricos = 0
            kg_prot_teoricos = 0
            envio_litros_teoricos = 0
        
        perdida_mg_dia = np.random.normal(media_perdida, sigma_perdida)
        perdida_mp_dia = np.random.normal(media_perdida, sigma_perdida)
        inc_agua_dia = np.random.normal(media_agua, sigma_agua)
        
        envio_litros_final = int(round(envio_litros_teoricos * (1 + inc_agua_dia)))
        
        kg_grasa_final = kg_grasa_teoricos * (1 - perdida_mg_dia)
        kg_prot_final = kg_prot_teoricos * (1 - perdida_mp_dia)
        
        if envio_litros_final > 0:
            envio_grasa_g_l = (kg_grasa_final * 1000) / envio_litros_final
            envio_prot_g_l = (kg_prot_final * 1000) / envio_litros_final
        
        stock_leche_litros -= envio_litros_teoricos
        stock_grasa_kg_total -= kg_grasa_teoricos
        stock_proteina_kg_total -= kg_prot_teoricos

    # --- GUARDAR FILA ---
    data.append({
        'fecha': fecha.strftime('%d/%m/%Y'),
        'dia_semana': dia_semana_es,
        'semana': (i // 7) + 1,
        'entrada_kg': int(entrada_kg),
        'entrada_litros': int(entrada_litros),
        'entrada_grasa_g_l': round(grasa_g_l, 2),
        'entrada_prot_g_l': round(prot_g_l, 2),
        'envio_fab_litros': int(envio_litros_final),
        'envio_fab_grasa_g_l': round(envio_grasa_g_l, 2),
        'envio_fab_prot_g_l': round(envio_prot_g_l, 2),
        'ratio_perdida_mg': round(perdida_mg_dia, 5),
        'ratio_perdida_mp': round(perdida_mp_dia, 5),
        'ratio_inc_agua': round(inc_agua_dia, 5),
        'stock_cierre_litros': int(round(max(0, stock_leche_litros)))
    })

df = pd.DataFrame(data)

# EXPORTACIÓN OPTIMIZADA PARA WINDOWS/EXCEL
# encoding='utf-8-sig' añade el BOM para que Excel reconozca los acentos
df.to_csv(nombre_fichero, index=False, sep=';', decimal=',', encoding='utf-8-sig')

print(f"Fichero '{nombre_fichero}' generado con éxito.")
print("Formato: UTF-8 con BOM (compatible con acentos en Windows Excel).")

Fichero 'queseria_final_tidy.csv' generado con éxito.
Formato: UTF-8 con BOM (compatible con acentos en Windows Excel).
